# 01 · Limpieza de Kardex y Balanzas

Reto 1 — Monitoreo de saldo de inventarios, FEMSA Salud Chile (FCV).

Este notebook prepara los datos para las alertas de conciliación, variación mensual y antigüedad. Las fuentes son dos reportes mensuales (enero a julio de 2026):

- **Kardex** (`INV_Kardex_AAAAMM.xlsx`): detalle del inventario por SKU al cierre de cada mes, con stock, costo, valorización y fecha de última compra (FULCOM).
- **Balanza de comprobación** (`AAAAMM_Balanza.xlsx`): saldos contables por cuenta. Para este reto solo interesa la cuenta de existencias, que en los datos anonimizados aparece como `CTA_000184`.

La salida son dos tablas en `data/processed/`: el Kardex de todos los meses en una sola tabla (`kardex_limpio.csv`) y el saldo contable de existencias por mes (`gl_existencias.csv`). No se aplica todavía ninguna regla de alerta; eso queda para el siguiente notebook.

Requisito: los Excel deben estar desencriptados en `data/raw/` (`dvc pull -r storage` y `./scripts/dvc_decrypt.sh`).

## 1. Configuración

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 30)


def raiz_del_repo():
    # el notebook puede abrirse desde notebooks/ o desde la raíz; buscamos pyproject.toml hacia arriba
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if (carpeta / "pyproject.toml").exists():
            return carpeta
    raise FileNotFoundError("No se encontró pyproject.toml; abrir el notebook dentro del repo")


ROOT = raiz_del_repo()
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

CUENTA_EXISTENCIAS = "CTA_000184"  # equivale a la 111080001 del planteamiento


def mes_del_nombre(path):
    m = re.search(r"(20\d{2}(0[1-9]|1[0-2]))", path.name)
    return m.group(1) if m else None

## 2. Archivos de entrada

Se identifica el periodo por el `AAAAMM` del nombre. La balanza de julio viene nombrada como `Blalanza`, así que el patrón contempla ambas escrituras.

In [ ]:
archivos_kardex = sorted(RAW.glob("*Kardex_*.xlsx"), key=mes_del_nombre)
archivos_balanza = sorted(
    [p for p in RAW.glob("*.xlsx") if re.search(r"B(a|la)lanza", p.name, re.I)],
    key=mes_del_nombre,
)

assert archivos_kardex, "No hay Kardex en data/raw (¿falta dvc pull / dvc_decrypt.sh?)"
assert [mes_del_nombre(p) for p in archivos_kardex] == [mes_del_nombre(p) for p in archivos_balanza], \
    "Los periodos de Kardex y Balanza no coinciden"

pd.DataFrame({
    "periodo": [mes_del_nombre(p) for p in archivos_kardex],
    "kardex": [p.name for p in archivos_kardex],
    "balanza": [p.name for p in archivos_balanza],
})

## 3. Kardex

Cada archivo tiene la hoja `DETALLE KARDEX` con una fila por SKU. Se conservan solo las columnas base; las columnas de alertas que trae el Excel (`ALERTA STOCK`, `ESTADO GENERAL`, etc.) se descartan porque se van a recalcular.

Al revisar los archivos se encontró que el de julio trae, debajo del detalle, cuatro filas sin código con subtotales (total Kardex, total GL y diferencias). Se eliminan todas las filas sin `CODIGO`.

In [ ]:
COLUMNAS_KARDEX = {
    "PERIODO": "PERIODO",
    "CODIGO": "CODIGO",
    "DESCRIPTOR": "DESCRIPTOR",
    "STOCK": "STOCK",
    "COSTO": "COSTO",
    "VALORIZACION": "VALORIZACION",
    "FULCOM": "FULCOM",
    "UNID CONV": "UNID_CONV",
    "MESES ANTIG.": "MESES_ANTIG_ORIGINAL",
    "ALERTA ANTIG.": "SEMAFORO_ORIGINAL",
    "LINEA": "LINEA",
    "NOMBRE LINEA": "NOMBRE_LINEA",
    "POLITICA CJE AL 30SEP": "POLITICA_CANJE",  # P1 con canje, P0 sin canje
}

partes, registro = [], []
for path in archivos_kardex:
    mes = mes_del_nombre(path)
    df = pd.read_excel(path, sheet_name="DETALLE KARDEX")
    sin_codigo = df["CODIGO"].isna()
    registro.append({
        "MES": mes,
        "filas_leidas": len(df),
        "filas_sin_codigo": int(sin_codigo.sum()),
        "valor_filas_sin_codigo": df.loc[sin_codigo, "VALORIZACION"].sum(),
    })
    df = df.loc[~sin_codigo, list(COLUMNAS_KARDEX)].rename(columns=COLUMNAS_KARDEX)
    df.insert(0, "MES", mes)
    partes.append(df)

kardex = pd.concat(partes, ignore_index=True)
pd.DataFrame(registro)

Solo julio tenía filas sin código. Su suma (~234 mil millones CLP) casi triplicaba el valor del mes, así que de no quitarlas la conciliación de julio habría salido con una diferencia enorme que no existe.

In [ ]:
kardex["PERIODO"] = pd.to_datetime(kardex["PERIODO"])
kardex["FECHA_CORTE"] = kardex["PERIODO"] + pd.offsets.MonthEnd(0)
kardex["FULCOM"] = pd.to_datetime(kardex["FULCOM"], errors="coerce").dt.normalize()  # algunas traen hora
kardex["LINEA"] = kardex["LINEA"].astype("Int64")

print(f"{len(kardex):,} filas (SKU × mes) | {kardex['CODIGO'].nunique():,} SKUs distintos")
kardex.head()

## 4. Balanzas: cuenta de existencias

La hoja `FCV` tiene siete filas de encabezado antes de los datos y las columnas están agrupadas por bloque (Altas, Balance, Inventario, Pérdida/Ganancia). El saldo de la cuenta se toma del bloque Balance:

$$\text{Saldo GL} = \text{Balance Débito} - \text{Balance Crédito}$$

Además de `CTA_000184` se extrae `CTA_000194 Existencias PT Terceriza` como referencia, porque está pendiente confirmar si forma parte de la conciliación.

In [ ]:
COLS_BALANZA = ["CUENTA", "DESCRIPCION",
                "ALTAS_DEBITO", "ALTAS_CREDITO",
                "BALANCE_DEBITO", "BALANCE_CREDITO",
                "INV_DEBITO", "INV_CREDITO",
                "PG_DEBITO", "PG_CREDITO", "MOVIMIENTO_MES"]

filas = []
for path in archivos_balanza:
    mes = mes_del_nombre(path)
    hoja = next(h for h in pd.ExcelFile(path).sheet_names if h.strip().upper() == "FCV")  # en enero es "FCV "
    bal = pd.read_excel(path, sheet_name=hoja, header=None, skiprows=7).iloc[:, :11]
    bal.columns = COLS_BALANZA
    for cuenta in [CUENTA_EXISTENCIAS, "CTA_000194"]:
        r = bal[bal["CUENTA"] == cuenta]
        assert len(r) == 1, f"{cuenta} no aparece una sola vez en {path.name}"
        r = r.iloc[0]
        filas.append({
            "MES": mes,
            "CUENTA": cuenta,
            "DESCRIPCION": r["DESCRIPCION"],
            "BALANCE_DEBITO": r["BALANCE_DEBITO"],
            "BALANCE_CREDITO": r["BALANCE_CREDITO"],
            "SALDO_GL": r["BALANCE_DEBITO"] - r["BALANCE_CREDITO"],
        })

gl_cuentas = pd.DataFrame(filas)
gl = gl_cuentas[gl_cuentas["CUENTA"] == CUENTA_EXISTENCIAS].reset_index(drop=True)
gl["PERIODO"] = pd.to_datetime(gl["MES"] + "01")
gl

In [ ]:
gl_cuentas.loc[gl_cuentas["CUENTA"] == "CTA_000194", ["MES", "DESCRIPCION", "SALDO_GL"]]

En los siete meses la cuenta de existencias tiene saldo deudor y crédito cero, es decir, no hay saldos de naturaleza contraria a nivel contable. `CTA_000194` es pequeña (8–11 millones CLP), del orden de 0.01% del saldo de existencias.

## 5. Antigüedad

El Kardex trae una columna `MESES ANTIG.`, pero no es consistente entre meses. Un ejemplo con un SKU cuya FULCOM no cambia en todo el periodo:

In [ ]:
kardex.loc[kardex["CODIGO"] == "SKU_006155",
           ["MES", "STOCK", "FULCOM", "MESES_ANTIG_ORIGINAL", "SEMAFORO_ORIGINAL"]]

Con la misma fecha de última compra (15-may-2025), la antigüedad debería crecer un mes por periodo. En cambio, en junio baja de 16 a 13, y en enero ya marca 12 meses cuando han pasado poco más de 8. No se sabe contra qué fecha se calculó, así que se recalcula desde la FULCOM:

$$\text{días de antigüedad} = \text{fecha de corte (fin de mes)} - \text{FULCOM}$$

Se guarda en días para no amarrarse a una política de semáforo; los rangos (0–30, 0–90, 0–180 días, etc.) se aplican después, cuando se confirme cuál es la vigente.

In [ ]:
kardex["DIAS_ANTIG"] = (kardex["FECHA_CORTE"] - kardex["FULCOM"]).dt.days
kardex["MESES_ANTIG"] = np.floor(kardex["DIAS_ANTIG"] / 30.44)

kardex.loc[kardex["CODIGO"] == "SKU_006155",
           ["MES", "FULCOM", "MESES_ANTIG_ORIGINAL", "DIAS_ANTIG", "MESES_ANTIG"]]

## 6. Calidad de datos

No se elimina ninguna partida con problemas porque eso descuadraría el total contra la balanza. En su lugar se agregan columnas booleanas para poder filtrarlas o reportarlas:

- `F_FECHA_FUTURA`: la FULCOM es posterior al cierre del mes (antigüedad negativa).
- `F_SIN_FECHA`: no hay FULCOM.
- `F_CONV_FALTANTE`: la unidad de conversión viene vacía (en el Excel aparece como `#N/A`).
- `F_CONV_DISTINTA_1`: la unidad de conversión es distinta de 1; hay que validar que el costo esté expresado en la misma unidad que el stock.
- `F_DUPLICADO`: el mismo SKU aparece más de una vez en el mismo mes.

In [ ]:
kardex["F_FECHA_FUTURA"] = kardex["DIAS_ANTIG"] < 0
kardex["F_SIN_FECHA"] = kardex["FULCOM"].isna()
kardex["F_CONV_FALTANTE"] = kardex["UNID_CONV"].isna()
kardex["F_CONV_DISTINTA_1"] = kardex["UNID_CONV"].notna() & (kardex["UNID_CONV"] != 1)
kardex["F_DUPLICADO"] = kardex.duplicated(subset=["MES", "CODIGO"], keep=False)

banderas = [c for c in kardex.columns if c.startswith("F_")]
kardex["DATO_OK"] = ~kardex[banderas].any(axis=1)

resumen_calidad = kardex.groupby("MES")[banderas + ["DATO_OK"]].sum()
resumen_calidad["PCT_FECHA_FUTURA"] = kardex.groupby("MES")["F_FECHA_FUTURA"].mean() * 100
resumen_calidad

In [ ]:
# lo mismo, pero en valor
(kardex.assign(VALOR_FECHA_FUTURA=kardex["VALORIZACION"].where(kardex["F_FECHA_FUTURA"], 0))
       .groupby("MES")[["VALORIZACION", "VALOR_FECHA_FUTURA"]].sum()
       .assign(PCT=lambda t: t["VALOR_FECHA_FUTURA"] / t["VALORIZACION"] * 100))

Lo más relevante de esta sección son las fechas futuras: alrededor del 70% de las partidas, y cerca del 98% del valor, tienen una FULCOM posterior al cierre del mes. En el Kardex de enero, por ejemplo, hay fechas hasta el 24 de mayo de 2026. En el Excel original estas partidas quedan con 0 meses y semáforo verde, lo que hace que el inventario se vea más sano de lo que podría ser.

Una posible explicación es que las fechas se hayan desplazado al anonimizar los datos, pero hay que confirmarlo con el socio antes de construir el semáforo y las matrices de transición, porque ambos dependen directamente de esta fecha.

Fuera de eso, los problemas son menores: unas 110 partidas por mes con conversión distinta de 1, dos con conversión vacía en algunos meses y ningún SKU duplicado una vez quitadas las filas de subtotales.

In [ ]:
# la valorización del Excel es STOCK * COSTO, así que esta comparación siempre debería dar cero
dif = kardex["STOCK"] * kardex["COSTO"] - kardex["VALORIZACION"]
print("Partidas con |stock × costo − valorización| > 1:", int((dif.abs() > 1).sum()))

Da cero, pero no es una validación real: la valorización se calcula con esa misma fórmula. Para que este control tuviera sentido haría falta un valor independiente del ERP.

## 7. Validación contra la propuesta de julio

La hoja *Dashboard Inventario Julio* del archivo de propuesta reporta 15,357 partidas, un detalle de Kardex de 116,975,118,671 CLP y un saldo GL de 117,495,620,727 CLP. Si los números coinciden, la carga y la limpieza son consistentes con lo que ya tiene el área.

In [ ]:
jul = kardex[kardex["MES"] == "202607"]
gl_jul = gl.loc[gl["MES"] == "202607", "SALDO_GL"].iloc[0]

print(f"Partidas:       {len(jul):>18,}")
print(f"Valor Kardex:   {jul['VALORIZACION'].sum():>18,.0f}")
print(f"Saldo GL:       {gl_jul:>18,.0f}")

assert len(jul) == 15_357
assert round(jul["VALORIZACION"].sum()) == 116_975_118_671
assert gl_jul == 117_495_620_727

Como referencia para el siguiente notebook, la comparación mes a mes entre el Kardex y la balanza:

In [ ]:
comparativo = (kardex.groupby("MES")
               .agg(SKUS=("CODIGO", "nunique"), VALOR_KARDEX=("VALORIZACION", "sum"))
               .reset_index()
               .merge(gl[["MES", "SALDO_GL"]], on="MES"))
comparativo["DIFERENCIA"] = comparativo["SALDO_GL"] - comparativo["VALOR_KARDEX"]
comparativo["DIF_PCT"] = comparativo["DIFERENCIA"] / comparativo["SALDO_GL"] * 100
comparativo["VAR_GL_PCT"] = comparativo["SALDO_GL"].pct_change() * 100
comparativo

La diferencia se mantiene entre 476 y 521 millones CLP (≈0.4% del saldo) todos los meses. Coincide con el renglón *RECETARIO* de la hoja `TD DETA`, que no forma parte del detalle por SKU; habrá que conseguir esa cifra como dato de entrada para la conciliación. En cuanto a la variación mensual, mayo es el único mes que supera el umbral de 5% (−7.35%).

## 8. Exportación

Se exporta en CSV con `utf-8-sig` para que Excel y Power BI lean bien los acentos. Estos archivos no se versionan en claro: se encriptan con `scripts/dvc_encrypt.sh` y se suben con DVC.

In [ ]:
columnas_salida = [
    "MES", "PERIODO", "FECHA_CORTE", "CODIGO", "DESCRIPTOR",
    "LINEA", "NOMBRE_LINEA", "POLITICA_CANJE",
    "STOCK", "COSTO", "VALORIZACION", "UNID_CONV",
    "FULCOM", "DIAS_ANTIG", "MESES_ANTIG",
    "MESES_ANTIG_ORIGINAL", "SEMAFORO_ORIGINAL",
    *banderas, "DATO_OK",
]

kardex[columnas_salida].to_csv(PROCESSED / "kardex_limpio.csv", index=False, encoding="utf-8-sig")
gl.to_csv(PROCESSED / "gl_existencias.csv", index=False, encoding="utf-8-sig")
resumen_calidad.to_csv(PROCESSED / "resumen_calidad.csv", encoding="utf-8-sig")

for f in sorted(PROCESSED.glob("*.csv")):
    print(f"{f.relative_to(ROOT)}  ({f.stat().st_size / 1e6:.1f} MB)")

## Pendientes con el socio

1. Origen de las FULCOM posteriores al cierre de mes (¿desplazamiento por anonimización?).
2. Política de semáforo vigente: el planteamiento menciona 0–30 días y más de un año; la propuesta usa 0–90 / 91–180 / >180 días; el Kardex usa ≤6 / 7–12 / >12 meses.
3. Fuente y detalle del recetario, que explica la diferencia Kardex vs. GL.
4. Si `CTA_000194 Existencias PT Terceriza` entra en la conciliación.
5. Moneda de reporte (CLP o MXN con el tipo de cambio de Banxico).